# PoC — Analyse EEG Motor Imagery
**UE28 Big Data — HELMo 2025-2026**

Dataset : EEG Motor Movement/Imagery (PhysioNet) — 66 sujets, 64 canaux, 160 Hz

## Plan
- **Partie A** : Conversion EDF → Parquet (Python + MNE)
- **Partie B** : Exploration avec Spark natif
- **Partie C** : Feature Engineering (puissance Alpha/Beta/Gamma)
- **Partie D** : Classification MLlib (RandomForest)
- **Partie E** : Dashboard Plotly

---
## Partie A — Conversion EDF → Parquet
### A1. Exploration d'un fichier EDF

Avant de tout convertir, on inspecte un seul fichier pour comprendre sa structure.

In [29]:
import mne
mne.set_log_level('WARNING')  # Réduire les messages verbeux

# Lire un seul fichier EDF
edf_path = '/opt/spark/data/eeg/S001/S001R03.edf'
raw = mne.io.read_raw_edf(edf_path, preload=True)

print('=== INFOS DU FICHIER ===')
print(f'Fréquence échantillonnage : {raw.info["sfreq"]} Hz')
print(f'Nombre de canaux          : {len(raw.ch_names)}')
print(f'Durée du signal           : {raw.times[-1]:.1f} secondes')
print(f'Nombre total de samples   : {len(raw.times)}')
print(f'\nNoms des canaux :')
print(raw.ch_names)

=== INFOS DU FICHIER ===
Fréquence échantillonnage : 160.0 Hz
Nombre de canaux          : 64
Durée du signal           : 125.0 secondes
Nombre total de samples   : 20000

Noms des canaux :
['Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.', 'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..', 'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.', 'Fp1.', 'Fpz.', 'Fp2.', 'Af7.', 'Af3.', 'Afz.', 'Af4.', 'Af8.', 'F7..', 'F5..', 'F3..', 'F1..', 'Fz..', 'F2..', 'F4..', 'F6..', 'F8..', 'Ft7.', 'Ft8.', 'T7..', 'T8..', 'T9..', 'T10.', 'Tp7.', 'Tp8.', 'P7..', 'P5..', 'P3..', 'P1..', 'Pz..', 'P2..', 'P4..', 'P6..', 'P8..', 'Po7.', 'Po3.', 'Poz.', 'Po4.', 'Po8.', 'O1..', 'Oz..', 'O2..', 'Iz..']


In [30]:
# Inspecter les annotations (les labels T0/T1/T2)
print('=== ANNOTATIONS (LABELS DE TÂCHES) ===')
print(raw.annotations)
print(f'\nNombre d événements : {len(raw.annotations)}')
print('\nDétail des 10 premiers événements :')
for ann in raw.annotations[:10]:
    print(f'  t={ann["onset"]:6.2f}s  durée={ann["duration"]:4.1f}s  label={ann["description"]}')

=== ANNOTATIONS (LABELS DE TÂCHES) ===
<Annotations | 30 segments: T0 (15), T1 (8), T2 (7)>

Nombre d événements : 30

Détail des 10 premiers événements :
  t=  0.00s  durée= 4.2s  label=T0
  t=  4.20s  durée= 4.1s  label=T2
  t=  8.30s  durée= 4.2s  label=T0
  t= 12.50s  durée= 4.1s  label=T1
  t= 16.60s  durée= 4.2s  label=T0
  t= 20.80s  durée= 4.1s  label=T1
  t= 24.90s  durée= 4.2s  label=T0
  t= 29.10s  durée= 4.1s  label=T2
  t= 33.20s  durée= 4.2s  label=T0
  t= 37.40s  durée= 4.1s  label=T2


### A2. Conversion d'un fichier EDF → DataFrame

On transforme le signal brut en tableau structuré avec les colonnes :
`subject_id | run_id | time | task_label | canal_1 | ... | canal_64`

In [31]:
import pandas as pd
import numpy as np

def edf_to_dataframe(edf_path, subject_id, run_id):
    """
    Lit un fichier EDF et retourne un DataFrame pandas structuré.
    Chaque ligne = 1 sample temporel (1/160 seconde).
    """
    raw = mne.io.read_raw_edf(edf_path, preload=True)
    
    # Extraire le signal : shape (n_canaux, n_samples)
    data, times = raw.get_data(return_times=True)
    
    # Construire le DataFrame : 1 colonne par canal
    df = pd.DataFrame(data.T, columns=raw.ch_names)  # Transposé : (n_samples, n_canaux)
    df.insert(0, 'time', times)
    df.insert(0, 'run_id', run_id)
    df.insert(0, 'subject_id', subject_id)
    
    # Associer chaque sample à son label de tâche (T0/T1/T2)
    df['task_label'] = 'T0'  # Par défaut repos
    for ann in raw.annotations:
        onset = ann['onset']
        end   = onset + ann['duration']
        label = ann['description']
        mask  = (df['time'] >= onset) & (df['time'] < end)
        df.loc[mask, 'task_label'] = label
    
    return df

# Test sur un seul fichier
df_test = edf_to_dataframe('/opt/spark/data/eeg/S001/S001R03.edf', 'S001', 'R03')

print(f'Shape : {df_test.shape}')
print(f'Colonnes : {list(df_test.columns[:8])} ...')
print(f'\nDistribution des labels :')
print(df_test['task_label'].value_counts())
df_test.head()

Shape : (20000, 68)
Colonnes : ['subject_id', 'run_id', 'time', 'Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.'] ...

Distribution des labels :
task_label
T0    10160
T1     5248
T2     4592
Name: count, dtype: int64


,subject_id,run_id,time,Fc5.,Fc3.,Fc1.,Fcz.,Fc2.,Fc4.,Fc6.,...,Po7.,Po3.,Poz.,Po4.,Po8.,O1..,Oz..,O2..,Iz..,task_label
0,S001,R03,0.00000,-0.000057,-0.000013,-0.000015,-0.000012,-0.000013,-0.000008,-0.000040,...,-0.000038,-0.000042,-0.000068,-0.000076,-0.000103,-0.000051,-0.000056,-0.000124,-0.000028,T0
1,S001,R03,0.00625,-0.000049,-0.000011,-0.000010,-0.000012,-0.000019,-0.000024,-0.000058,...,-0.000055,-0.000063,-0.000082,-0.000087,-0.000099,-0.000059,-0.000070,-0.000149,-0.000040,T0
2,S001,R03,0.01250,-0.000055,-0.000017,-0.000016,-0.000019,-0.000024,-0.000029,-0.000066,...,-0.000063,-0.000072,-0.000091,-0.000092,-0.000091,-0.000067,-0.000077,-0.000153,-0.000037,T0
3,S001,R03,0.01875,-0.000073,-0.000042,-0.000040,-0.000037,-0.000037,-0.000040,-0.000071,...,-0.000052,-0.000066,-0.000100,-0.000105,-0.000105,-0.000067,-0.000072,-0.000148,-0.000026,T0
4,S001,R03,0.02500,-0.000087,-0.000053,-0.000052,-0.000051,-0.000045,-0.000043,-0.000071,...,-0.000082,-0.000090,-0.000117,-0.000119,-0.000118,-0.000075,-0.000082,-0.000161,-0.000035,T0


### A3. Conversion complète EDF → Parquet

On applique la conversion à tous les sujets et on sauvegarde en Parquet.
**Attention** : cette cellule prend ~5-10 minutes.

In [32]:
import os
import glob

EDF_DIR     = '/opt/spark/data/eeg/'
PARQUET_DIR = '/opt/spark/data/parquet/'
os.makedirs(PARQUET_DIR, exist_ok=True)

# Runs moteurs uniquement (R03-R14 contiennent T0/T1/T2 — R01/R02 sont des refs)
MOTOR_RUNS = [f'R{i:02d}' for i in range(3, 15)]

converted = 0
errors    = 0

subjects = sorted(os.listdir(EDF_DIR))
print(f'Sujets trouvés : {len(subjects)}')

for subject in subjects:
    subject_dir = os.path.join(EDF_DIR, subject)
    if not os.path.isdir(subject_dir):
        continue
    
    for run in MOTOR_RUNS:
        edf_file    = os.path.join(subject_dir, f'{subject}{run}.edf')
        parquet_file = os.path.join(PARQUET_DIR, f'{subject}_{run}.parquet')
        
        if not os.path.exists(edf_file):
            continue
        if os.path.exists(parquet_file):
            continue  # Déjà converti
        
        try:
            df = edf_to_dataframe(edf_file, subject, run)
            df.to_parquet(parquet_file, index=False)
            converted += 1
        except Exception as e:
            errors += 1
            print(f'Erreur {subject}/{run} : {e}')
    
    print(f'{subject} converti ({converted} fichiers au total)', end='\r')

print(f'\n✓ Conversion terminée : {converted} fichiers Parquet créés, {errors} erreurs')

Sujets trouvés : 66
S066 converti (0 fichiers au total)
✓ Conversion terminée : 0 fichiers Parquet créés, 0 erreurs


---
## Partie B — Exploration avec Spark
### B1. Création de la SparkSession

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('EEG-PoC') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '8g') \
    .config('spark.driver.memory', '4g') \
    .getOrCreate()

print('SparkSession créée')
print(f'Version Spark : {spark.version}')
print(f'UI disponible : http://localhost:4040')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/17 18:10:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession créée
Version Spark : 3.5.5
UI disponible : http://localhost:4040


### B2. Lecture des Parquet et exploration du schéma

In [2]:
# Lire tous les Parquet d'un coup
df = spark.read.parquet('/opt/spark/data/parquet/')

print('=== SCHÉMA ===')
df.printSchema()

print('\n=== DIMENSIONS ===')
print(f'Lignes   : {df.count():,}')
print(f'Colonnes : {len(df.columns)}')

=== SCHÉMA ===
root
 |-- subject_id: string (nullable = true)
 |-- run_id: string (nullable = true)
 |-- time: double (nullable = true)
 |-- Fc5.: double (nullable = true)
 |-- Fc3.: double (nullable = true)
 |-- Fc1.: double (nullable = true)
 |-- Fcz.: double (nullable = true)
 |-- Fc2.: double (nullable = true)
 |-- Fc4.: double (nullable = true)
 |-- Fc6.: double (nullable = true)
 |-- C5..: double (nullable = true)
 |-- C3..: double (nullable = true)
 |-- C1..: double (nullable = true)
 |-- Cz..: double (nullable = true)
 |-- C2..: double (nullable = true)
 |-- C4..: double (nullable = true)
 |-- C6..: double (nullable = true)
 |-- Cp5.: double (nullable = true)
 |-- Cp3.: double (nullable = true)
 |-- Cp1.: double (nullable = true)
 |-- Cpz.: double (nullable = true)
 |-- Cp2.: double (nullable = true)
 |-- Cp4.: double (nullable = true)
 |-- Cp6.: double (nullable = true)
 |-- Fp1.: double (nullable = true)
 |-- Fpz.: double (nullable = true)
 |-- Fp2.: double (nullable = true)


Lignes   : 15,264,320
Colonnes : 68


### B3. Stats descriptives

In [3]:
from pyspark.sql.functions import count, col

# Distribution des labels par sujet
print('=== DISTRIBUTION DES TÂCHES ===')
df.groupBy('task_label').count().orderBy('task_label').show()

print('=== SUJETS DISPONIBLES ===')
df.groupBy('subject_id').count().orderBy('subject_id').show(10)

=== DISTRIBUTION DES TÂCHES ===


+----------+-------+
|task_label|  count|
+----------+-------+
|        T0|7658928|
|        T1|3817776|
|        T2|3787616|
+----------+-------+

=== SUJETS DISPONIBLES ===


+----------+------+
|subject_id| count|
+----------+------+
|      S001|240000|
|      S002|236160|
|      S003|240000|
|      S004|236160|
|      S005|236160|
|      S006|236160|
|      S007|240000|
|      S008|236160|
|      S009|236160|
|      S010|236160|
+----------+------+
only showing top 10 rows



In [4]:
from pyspark.sql.functions import col

canaux_moteurs = ['C3..', 'Cz..', 'C4..']

df.select([
    col(f'`{c}`').alias(c.replace('.', ''))
    for c in canaux_moteurs
]).describe().show()

+-------+--------------------+--------------------+--------------------+
|summary|                  C3|                  Cz|                  C4|
+-------+--------------------+--------------------+--------------------+
|  count|            15264320|            15264320|            15264320|
|   mean|-1.47546657826879...|-1.32957589987633...|-3.49044896857509...|
| stddev|6.325278671774448E-5|5.472000674456126E-5|5.721932827387137E-5|
|    min|-6.74999999999999...|            -5.92E-4|-6.52999999999999...|
|    max|             6.86E-4|             6.21E-4|             6.29E-4|
+-------+--------------------+--------------------+--------------------+



---
## Partie C — Feature Engineering
### C1. Découpage en epochs et calcul de puissance spectrale

On calcule la puissance dans chaque bande de fréquence par canal :
- **Alpha** : 8–13 Hz → diminue pendant l'imagerie motrice
- **Beta**  : 13–30 Hz → augmente pendant l'imagerie motrice  
- **Gamma** : 30–100 Hz → traitement cognitif

In [5]:
import numpy as np
from pyspark.sql.functions import pandas_udf, col, collect_list
from pyspark.sql.types import DoubleType, StructType, StructField, ArrayType
import pandas as pd

FS = 160  # Fréquence d'échantillonnage en Hz

def band_power(signal, fs, low, high):
    """Calcule la puissance d'un signal dans une bande de fréquence."""
    freqs = np.fft.rfftfreq(len(signal), 1.0 / fs)
    fft   = np.abs(np.fft.rfft(signal)) ** 2
    mask  = (freqs >= low) & (freqs < high)
    return float(np.mean(fft[mask])) if mask.any() else 0.0

print('Fonction band_power définie.')
print('Elle utilise la FFT (transformée de Fourier) pour décomposer le signal en fréquences.')

Fonction band_power définie.
Elle utilise la FFT (transformée de Fourier) pour décomposer le signal en fréquences.


In [6]:
from pyspark.sql.functions import floor, col

# Renommer toutes les colonnes pour supprimer les points
for c in df.columns:
    if '.' in c:
        df = df.withColumnRenamed(c, c.replace('.', ''))

# Définir les colonnes
meta_cols  = ['subject_id', 'run_id', 'time', 'task_label']
canal_cols = [c for c in df.columns if c not in meta_cols]

# Créer les epochs (fenêtres de 2 secondes)
EPOCH_SEC = 2
FS = 160
df_epochs = df.withColumn('epoch_id', floor(col('time') / EPOCH_SEC).cast('int'))

print(f'Colonnes renommées. Exemple : {canal_cols[:5]}')
print(f'Canaux EEG : {len(canal_cols)}')
print(f'Taille epoch : {EPOCH_SEC}s × {FS}Hz = {EPOCH_SEC*FS} samples')
df_epochs.select('subject_id', 'run_id', 'epoch_id', 'task_label').show(5)

Colonnes renommées. Exemple : ['Fc5', 'Fc3', 'Fc1', 'Fcz', 'Fc2']
Canaux EEG : 64
Taille epoch : 2s × 160Hz = 320 samples
+----------+------+--------+----------+
|subject_id|run_id|epoch_id|task_label|
+----------+------+--------+----------+
|      S059|   R07|       0|        T0|
|      S059|   R07|       0|        T0|
|      S059|   R07|       0|        T0|
|      S059|   R07|       0|        T0|
|      S059|   R07|       0|        T0|
+----------+------+--------+----------+
only showing top 5 rows



In [7]:
import pyspark.sql.functions as F
from pyspark.sql.functions import first
from pyspark.sql.types import DoubleType

# Canaux moteurs uniquement — cortex moteur (C3=main droite, C4=main gauche, Cz=central)
# On ne calcule PAS les 64 canaux — ça prendrait 20x plus longtemps pour rien
motor_channels = ['C3', 'Cz', 'C4']

agg_exprs = [first('task_label').alias('task_label')]
for canal in motor_channels:
    for band_name, (low, high) in [('alpha', (8, 13)), ('beta', (13, 30)), ('gamma', (30, 80))]:
        agg_exprs.append(
            F.udf(lambda vals, l=low, h=high: band_power(np.array(vals), FS, l, h),
                  DoubleType())(F.collect_list(canal)).alias(f'{canal}_{band_name}')
        )

df_features = df_epochs.groupBy('subject_id', 'run_id', 'epoch_id').agg(*agg_exprs)

print('Calcul des features en cours (3 canaux × 3 bandes = 9 features)...')
df_features.cache()
n = df_features.count()
print(f'Features calculées : {n:,} epochs × {len(motor_channels)*3} features')
df_features.show(5)

Calcul des features en cours (3 canaux × 3 bandes = 9 features)...


Features calculées : 48,069 epochs × 9 features
+----------+------+--------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|subject_id|run_id|epoch_id|task_label|            C3_alpha|             C3_beta|            C3_gamma|            Cz_alpha|             Cz_beta|            Cz_gamma|            C4_alpha|             C4_beta|            C4_gamma|
+----------+------+--------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|      S001|   R04|      37|        T2|  6.1606069187102E-7|3.106512307101162...|5.199386740732073E-8|5.442003656817501E-7| 2.43634301581973E-7|4.633458214197868...|3.944379678163897...|1.823771948560849...|4.497308805346488E-8|
|      S001|   R13|       1|        

---
## Partie D — Classification MLlib
### D1. Préparation des features

In [8]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline
import pyspark.sql.functions as F

# Les 9 features des canaux moteurs (C3/Cz/C4 × alpha/beta/gamma)
feature_cols = [c for c in df_features.columns
                if c not in ['subject_id', 'run_id', 'epoch_id', 'task_label']]

print(f'Features ({len(feature_cols)}) : {feature_cols}')

# Poids inversement proportionnel à la fréquence : T0=50% → 0.5, T1/T2=25% → 1.0
df_features_w = df_features.withColumn('weight',
    F.when(F.col('task_label') == 'T0', 0.5).otherwise(1.0)
)

indexer   = StringIndexer(inputCol='task_label', outputCol='label')
assembler = VectorAssembler(inputCols=feature_cols, outputCol='features')

print('Poids : T0=0.5, T1=1.0, T2=1.0')
print('Prêt pour D2 → split train/test')

Features (9) : ['C3_alpha', 'C3_beta', 'C3_gamma', 'Cz_alpha', 'Cz_beta', 'Cz_gamma', 'C4_alpha', 'C4_beta', 'C4_gamma']
Poids : T0=0.5, T1=1.0, T2=1.0
Prêt pour D2 → split train/test


In [9]:
# Split train/test 80/20
train_df, test_df = df_features_w.randomSplit([0.8, 0.2], seed=42)
print(f'Train : {train_df.count():,} epochs')
print(f'Test  : {test_df.count():,} epochs')

Train : 38,413 epochs


Test  : 9,656 epochs


In [10]:
# Entraîner le RandomForest avec poids de classe
rf = RandomForestClassifier(
    featuresCol='features',
    labelCol='label',
    weightCol='weight',
    numTrees=100,
    maxDepth=10,
    seed=42
)

pipeline = Pipeline(stages=[indexer, assembler, rf])

print('Entraînement du RandomForest (100 arbres, poids équilibrés)...')
model = pipeline.fit(train_df)
print('Modèle entraîné !')

Entraînement du RandomForest (100 arbres, poids équilibrés)...


26/04/17 18:13:03 WARN DAGScheduler: Broadcasting large task binary with size 1038.0 KiB
26/04/17 18:13:07 WARN DAGScheduler: Broadcasting large task binary with size 1801.6 KiB
26/04/17 18:13:12 WARN DAGScheduler: Broadcasting large task binary with size 3.1 MiB
26/04/17 18:13:22 WARN DAGScheduler: Broadcasting large task binary with size 5.1 MiB
26/04/17 18:13:34 WARN DAGScheduler: Broadcasting large task binary with size 1098.0 KiB
26/04/17 18:13:38 WARN DAGScheduler: Broadcasting large task binary with size 8.2 MiB
26/04/17 18:13:56 WARN DAGScheduler: Broadcasting large task binary with size 1614.6 KiB


Modèle entraîné !


In [11]:
# Évaluation
predictions = model.transform(test_df)

evaluator = MulticlassClassificationEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='accuracy'
)

accuracy = evaluator.evaluate(predictions)
print(f'Accuracy : {accuracy:.2%}')
print(f'(baseline aléatoire 3 classes : 33%)')

# Matrice de confusion
predictions.groupBy('task_label', 'prediction').count().orderBy('task_label').show()

26/04/17 18:18:16 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB


Accuracy : 41.82%
(baseline aléatoire 3 classes : 33%)


26/04/17 18:18:26 WARN DAGScheduler: Broadcasting large task binary with size 6.7 MiB
26/04/17 18:18:33 WARN DAGScheduler: Broadcasting large task binary with size 6.6 MiB


+----------+----------+-----+
|task_label|prediction|count|
+----------+----------+-----+
|        T0|       1.0| 1494|
|        T0|       0.0| 2677|
|        T0|       2.0|  766|
|        T1|       1.0|  889|
|        T1|       0.0| 1030|
|        T1|       2.0|  426|
|        T2|       1.0|  803|
|        T2|       2.0|  472|
|        T2|       0.0| 1099|
+----------+----------+-----+



---
## Partie E — Dashboard Plotly

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pyspark.sql.functions as F

# E1. Signal brut du canal Cz dans le temps
df_plot = df.filter(
    (F.col('subject_id') == 'S001') & (F.col('run_id') == 'R03')
).orderBy('time').limit(1600).toPandas()

fig = px.line(
    df_plot,
    x='time', y='Cz',
    color='task_label',
    title='Signal EEG canal Cz — Sujet S001 Run R03 (10 premières secondes)',
    labels={'time': 'Temps (s)', 'Cz': 'Amplitude (V)'}
)

# Sauvegarder en HTML (Jupyter Docker ne rend pas toujours les graphes inline)
fig.write_html('/opt/spark/data/e1_signal.html')
print('E1 sauvegardé : /opt/spark/data/e1_signal.html')
fig.show()

In [ ]:
# E2. Matrice de confusion visuelle
import pandas as pd

# Mapper les prédictions numériques (0.0/1.0/2.0) vers les labels originaux
# StringIndexer assigne 0 à la classe la plus fréquente (T0), puis T1, T2
label_map = {0.0: 'T0', 1.0: 'T1', 2.0: 'T2'}

conf_pd = predictions.groupBy('task_label', 'prediction') \
    .count().toPandas()

conf_pd['prediction_label'] = conf_pd['prediction'].map(label_map)

matrix = pd.crosstab(
    conf_pd['task_label'],
    conf_pd['prediction_label'],
    values=conf_pd['count'],
    aggfunc='sum'
).fillna(0).reindex(index=['T0','T1','T2'], columns=['T0','T1','T2'], fill_value=0)

fig2 = px.imshow(
    matrix,
    text_auto=True,
    title='Matrice de confusion — RandomForest (C3/Cz/C4, poids équilibrés)',
    labels=dict(x='Prédit', y='Réel', color='Count'),
    color_continuous_scale='Blues'
)

fig2.write_html('/opt/spark/data/e2_confusion.html')
print('E2 sauvegardé : /opt/spark/data/e2_confusion.html')
fig2.show()

In [ ]:
# E3. Puissance Alpha par canal moteur (C3, Cz, C4) — comparaison T0/T1/T2
from pyspark.sql.functions import avg

motor_alpha_cols = ['C3_alpha', 'Cz_alpha', 'C4_alpha']

alpha_by_task = df_features.groupBy('task_label') \
    .agg(*[avg(c).alias(c) for c in motor_alpha_cols]) \
    .toPandas() \
    .melt(id_vars='task_label', var_name='canal', value_name='puissance_alpha')

alpha_by_task['canal'] = alpha_by_task['canal'].str.replace('_alpha', '')

fig3 = px.bar(
    alpha_by_task,
    x='canal', y='puissance_alpha',
    color='task_label',
    barmode='group',
    title='Puissance Alpha moyenne (C3/Cz/C4) par tâche',
    labels={'canal': 'Canal EEG', 'puissance_alpha': 'Puissance Alpha (µV²)', 'task_label': 'Tâche'},
    color_discrete_map={'T0': '#636EFA', 'T1': '#EF553B', 'T2': '#00CC96'}
)

fig3.write_html('/opt/spark/data/e3_alpha.html')
print('E3 sauvegardé : /opt/spark/data/e3_alpha.html')
print('\nInterprétation attendue :')
print('  T0 (repos)      → Alpha élevé (cerveau au repos)')
print('  T1/T2 (imagerie) → Alpha réduit sur C3 ou C4 (selon la main imaginée)')
fig3.show()

In [47]:
spark.stop()
print('SparkSession arrêtée.')

SparkSession arrêtée.
